In [8]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet101
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import time

In [9]:
# Constants / path
data_dir = "image-results-spoofing/data"
train_dir = os.path.join(data_dir, "Train")
test_dir = os.path.join(data_dir, "Test")
batch_size = 32
learning_rate = 0.001
num_epochs = 20
RANDOM_STATE = 1082
criterion = nn.CrossEntropyLoss()
MODEL_NAME = "ResNet-CNN"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../2D_Results.csv"

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [11]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

In [12]:
# Datasets and DataLoaders
train_full = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_full = datasets.ImageFolder(root=train_dir, transform=test_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_idx, val_idx = train_test_split(
    np.arange(len(train_full)),
    train_size=0.8,
    random_state=RANDOM_STATE,
    stratify=np.array(train_full.targets)
)
train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}
classes = train_full.classes


In [13]:
# Model
class CNN_Model(nn.Module):
    def __init__(self, num_classes=2, feature_dim=300):
        super(CNN_Model, self).__init__()
        self.resnet = resnet101(pretrained=True)
        # Change first conv layer for grayscale
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, feature_dim)
        self.fc1 = nn.Linear(feature_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        features = self.resnet(x)
        x = F.relu(self.fc1(features))
        x = self.fc2(x)
        return x

In [14]:
# Instantiate model, loss, optimizer
model = CNN_Model(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

In [15]:
# Training loop
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0
best_epoch = 0

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    all_train_preds = []
    all_train_labels = []
    for inputs, labels in dataloaders['train']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        all_train_preds.append(preds.cpu().numpy())
        all_train_labels.append(labels.cpu().numpy())
    epoch_loss = running_loss / len(train_dataset)
    train_preds = np.concatenate(all_train_preds)
    train_labels = np.concatenate(all_train_labels)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation
    model.eval()
    all_val_preds = []
    all_val_labels = []
    with torch.no_grad():
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_val_preds.append(preds.cpu().numpy())
            all_val_labels.append(labels.cpu().numpy())
    val_preds = np.concatenate(all_val_preds)
    val_labels = np.concatenate(all_val_labels)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

train_time = time.time() - start_time

model.load_state_dict(best_model_wts)

KeyboardInterrupt: 

In [ ]:
# Final evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference_torch


import numpy as np

_timing = time_inference_torch(model, next(iter(dataloaders['val']))[0].to(device))

row = evaluate(
    all_labels,
    all_preds,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="Image",
    labels=list(range(len(classes))),
    class_names=classes,
    y_proba=all_probs,
    y_train=np.array(train_full.targets)[train_idx],
    train_time_sec=train_time,
    num_epochs=num_epochs,
    parameters=list(dict(model.named_parameters()).keys()),
    seed=RANDOM_STATE,
    n_features=int(np.prod(train_dataset[0][0].shape)),
    n_train=len(train_idx),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")

In [ ]:
# Save best model
model_save_path = "best_resnet_model.pth"
checkpoint = {
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}
torch.save(checkpoint, model_save_path)
print(f"Model saved to {model_save_path}")